# Olist first-order profiling and repeat-order EDA

Run dbt first. This notebook reads the PostgreSQL feature mart through shared Arrow loaders and reuses the dbt-owned eligibility census. It writes reproducible summaries and plots, without reimplementing targets or exclusions. Set `OLIST_EDA_OUTPUT_DIR` to choose an artifact directory. Reports generated by `scripts/generate_data_profile.py` include all nine raw tables and the feature mart.

In [ ]:
from pathlib import Path
import os
import sys
import json
from IPython.display import display, Image
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'scripts' / 'profiling_analysis.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from profiling_analysis import generate_eda
from upload_data import public_error
OUTPUT_DIR = Path(os.getenv('OLIST_EDA_OUTPUT_DIR', str(PROJECT_ROOT / 'reports' / 'eda')))
try:
    analysis = generate_eda(OUTPUT_DIR)
except ValueError:
    raise
except Exception as exc:
    raise RuntimeError(public_error(exc)) from None
print(json.dumps(analysis, indent=2, default=str))

## Delivered counts and new target

Lifetime delivered-order counts are descriptive and are not predictors or the target. Repeat prevalence uses known labels only, including early observed positives. Uncertain customers remain a separate cohort.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'plots' / 'delivered_counts.png')))
display(Image(filename=str(OUTPUT_DIR / 'plots' / 'target_distribution.png')))

## First-order spend and state repeat rates

Spend compares known positive and negative labels. The top five states use labeled customer counts, ties alphabetical. Each state shows its known-label denominator. Rates measure observed historical repeat activity, not causal state effects.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'plots' / 'spend_by_target.png')))
display(Image(filename=str(OUTPUT_DIR / 'plots' / 'state_repeat_rates.png')))

## Numeric predictors and uncertain customers

Correlations use the eight numeric anchor predictors across all eligible customers. Identifiers, timestamps, target metadata and lifetime counts are excluded. Constant-column correlations are undefined. Uncertain spend is summarized separately. There is no classification evaluation against unknown labels.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'plots' / 'numeric_correlations.png')))
display(Image(filename=str(OUTPUT_DIR / 'plots' / 'uncertain_spend.png')))

## Interpretation and selection effects

Early observed positives enter the labeled cohort before complete follow-up, favoring fast observed repeats in recent cohorts. Unfinished customers without observed repeats remain uncertain. Retained delivery and review predictors may arrive after repeat activity. These historical associations do not establish prospective performance or causation. The default maximum recorded purchase timestamp assumes observation coverage and does not prove it. The census records original exclusions, overlapping warning reasons and exclusive patterns. No models are fitted or selected here.